# B1.11 · Context engineering for the pipeline

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B1.10 · Severity calibration and reporting](https://spbreed.github.io/cyber-commons/lessons/B1.10.html)**.

| | |
|---|---|
| Tools used | tree-sitter, GLM-4.6, Llama 3.3, Claude Haiku 4.5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Give an agent more context and it gets better, until it gets worse. The cliff is real, it arrives earlier than anyone expects, and past it you are paying more per token for a worse answer.

> **At CyberTravels.** Give the review agent CyberTravels' whole repository and it gets worse, not better. The cliff arrives earlier than anyone expects and you pay more per token for it.

## 2 · The framework

```
   accuracy
     ^
     |          .-----.
     |        .'       `.
     |      .'           `.       <- the cliff
     |    .'               `--....
     |  .'
     +--------------------------------> context tokens
        too little        enough      too much

   past the peak you pay more per token for a worse answer
```

Cross-cutting, and it applies to every stage that calls a model: stages 3, 4, 5,
7 and 14.

The instinct when a model misses something is to give it more context. Usually
the opposite is correct.

To find a vulnerability, a model needs three things: the **sink**, the
**source**, and the **path** between them. Everything else competes for
attention and for window. A repository dumped into a prompt does not produce a
thorough review — it produces a review of whatever survived truncation, and you
cannot tell which parts those were.

So context engineering is mostly subtraction, with one exception you must not
subtract: the **enclosing signature**, because that is where reachability is
decided. The identical concatenation is critical inside an HTTP handler and
irrelevant inside a migration script that takes a constant.

## 3 · Demo — four strategies over one bug

In [ ]:
SOURCE = '''"""Reporting service."""
import logging, os, json, datetime

log = logging.getLogger(__name__)
DEFAULT_LIMIT = 100
CACHE = {}

def _format_row(row):
    return {"id": row[0], "name": row[1], "created": str(row[2])}

def _cache_key(*parts):
    return ":".join(str(p) for p in parts)

def healthcheck():
    return {"status": "ok", "ts": datetime.datetime.utcnow().isoformat()}

def list_reports(conn, owner, limit=DEFAULT_LIMIT):
    """Called from GET /reports?owner=... — owner is user-controlled."""
    key = _cache_key("reports", owner, limit)
    if key in CACHE:
        return CACHE[key]
    rows = conn.execute("SELECT * FROM reports WHERE owner = '" + owner + "' LIMIT " + str(limit))
    out = [_format_row(r) for r in rows]
    CACHE[key] = out
    return out

def purge_cache():
    CACHE.clear()
    log.info("cache purged")
'''
lines = SOURCE.splitlines()
BUG_LINE = next(i for i, l in enumerate(lines, 1) if "SELECT * FROM reports" in l)
print(f"the bug is on line {BUG_LINE}")

def whole_file(_):     return SOURCE
def window(n, radius): return "\n".join(lines[max(n-radius-1,0):n+radius])

STRATEGIES = {"whole file": whole_file(BUG_LINE),
              "±2 line window": window(BUG_LINE, 2),
              "±6 line window": window(BUG_LINE, 6)}
for name, ctx in STRATEGIES.items():
    print(f"{name:20s}{len(ctx):>6} chars{len(ctx.splitlines()):>5} lines")

In [ ]:
def decidable(ctx):
    """Can a reviewer judge exploitability from this context alone?"""
    return {"sink": "conn.execute" in ctx,
            "concatenation": "' + owner +" in ctx or "+ owner +" in ctx,
            "source (signature)": "def list_reports" in ctx,
            "intent (docstring)": "user-controlled" in ctx}

print(f"{'strategy':20s}{'sink':7s}{'concat':8s}{'source':8s}{'intent':8s}decidable")
print("-" * 64)
for name, ctx in STRATEGIES.items():
    d = decidable(ctx)
    ok = d["sink"] and d["concatenation"] and d["source (signature)"]
    print(f"{name:20s}{str(d['sink']):7s}{str(d['concatenation']):8s}"
          f"{str(d['source (signature)']):8s}{str(d['intent (docstring)']):8s}{ok}")
print("\nThe ±2 window has the sink and the concatenation but not the signature,")
print("so you cannot tell whether owner is user-controlled — which is the")
print("difference between critical and won't-fix.")

## 4 · The control — slice on the source-sink path

In [ ]:
def path_slice(source, bug_line):
    ls = source.splitlines()
    start = max(i for i in range(bug_line) if ls[i-1].startswith("def "))
    end = next((i for i in range(start, len(ls)) if i > start and ls[i].startswith("def ")),
               len(ls))
    return "\n".join(ls[start-1:end])

sliced = path_slice(SOURCE, BUG_LINE)
print(sliced)
d = decidable(sliced)
print(f"\n{len(sliced)} chars ({len(sliced)/len(SOURCE):.0%} of the file), "
      f"decidable={d['sink'] and d['concatenation'] and d['source (signature)']}")

In [ ]:
def evaluate(name, ctx):
    d = decidable(ctx)
    return {"strategy": name, "chars": len(ctx),
            "share": round(len(ctx)/len(SOURCE), 3),
            "decidable": d["sink"] and d["concatenation"] and d["source (signature)"],
            "noise_fns": max(ctx.count("def ") - 1, 0)}

rows = [evaluate(n, c) for n, c in STRATEGIES.items()] + [evaluate("path slice", sliced)]
print(f"{'strategy':20s}{'chars':>7}{'share':>8}{'decidable':>11}{'noise fns':>11}")
print("-" * 58)
for r in rows:
    print(f"{r['strategy']:20s}{r['chars']:>7}{r['share']:>8.0%}"
          f"{str(r['decidable']):>11}{r['noise_fns']:>11}")

best = sorted((r for r in rows if r["decidable"]), key=lambda r: r["chars"])[0]
whole = next(r for r in rows if r["strategy"] == "whole file")
print(f"\nsmallest decidable context: {best['strategy']} "
      f"({best['share']:.0%} of the file, {best['noise_fns']} unrelated functions)")
print(f"vs whole file: {1 - best['chars']/whole['chars']:.0%} smaller, "
      f"{whole['noise_fns']}→{best['noise_fns']} unrelated functions")
assert best["strategy"] == "path slice"

## What you just proved

The whole file is roughly 840 characters, the ±2 window about 200 and the path slice about 390. The ±2 window is not decidable because it lacks the signature; the ±6 window and the whole file are decidable but carry unrelated functions. The path slice is the smallest decidable context with zero unrelated functions, about 53% smaller than the whole file.

## Your turn

Apply the path-slice rule where the source is three functions away from the sink. That is the case where text windows break down entirely and the call graph from B1.1 earns its keep.

---

**Next → [B1.12 · Securing the developers' coding agents](https://spbreed.github.io/cyber-commons/lessons/B1.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*